### Question 1
List 3 differences between Reinforcement Learning, Supervised Learning, and Unsupervised Learning in a table, using real examples from apps like Instagram (feed ranking), Spotify (music recommendations), and chess game AI.

| Feature / Dimension | Supervised Learning | Unsupervised Learning | Reinforcement Learning |
| :--- | :--- | :--- | :--- |
| **1. Data & Learning Signal** | Learns from historical, labeled datasets with explicit input-output pairs $(X \rightarrow Y)$. | Discovers intrinsic patterns, groupings, or representations from unlabeled data $(X)$. | Learns through trial-and-error by interacting with an environment via states, actions, and scalar rewards. |
| **2. Feedback Mechanism** | Immediate ground-truth evaluation (error/loss computed directly per training example). | No external feedback or ground truth; optimizes statistical criteria (e.g., intra-cluster variance). | Evaluative and delayed numerical reward; current decisions affect future states and payoffs. |
| **3. Real-World Application Example** | **Instagram Feed Ranking:** Predicts the probability of a user liking or commenting on a specific post based on labeled past interactions (Liked = 1, Skipped = 0). | **Spotify Music Recommendations:** Groups tracks and listener listening profiles into latent acoustic clusters (tempo, energy, valence) without human genre tags. | **Chess Game AI (e.g., AlphaZero / Stockfish):** Selects moves sequentially to maximize the end-of-game probability of winning (Reward: +1 Win, 0 Draw, -1 Loss). |

In [9]:
import pandas as pd

comparison_data = {
    "Paradigm": ["Supervised Learning", "Unsupervised Learning", "Reinforcement Learning"],
    "Learning Signal": ["Historical labeled pairs (X -> Y)", "Unlabeled data patterns (X)", "State-action-reward interactions"],
    "Feedback Timing": ["Immediate target loss", "No explicit ground truth", "Delayed scalar reward"],
    "Example": ["Instagram Feed Ranking", "Spotify Song Clustering", "Chess Game AI"]
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison

,Paradigm,Learning Signal,Feedback Timing,Example
0,Supervised Learning,Historical labeled pairs (X -> Y),Immediate target loss,Instagram Feed Ranking
1,Unsupervised Learning,Unlabeled data patterns (X),No explicit ground truth,Spotify Song Clustering
2,Reinforcement Learning,State-action-reward interactions,Delayed scalar reward,Chess Game AI


### Question 2
Pick any one real-world application of Reinforcement Learning (such as game AI, robotics, self-driving cars, or recommendation systems) and explain step-by-step how an agent, environment, actions, and rewards would work in that scenario.

In a food ordering platform like Zomato, an RL agent balances immediate order conversion with kitchen latency, courier logistics, and long-term customer retention.

1. **Agent:** The centralized recommendation policy server operating in Zomato's ranking pipeline.
2. **Environment:** The user session, restaurant catalog, kitchen preparation backlogs, local courier availability, and external factors (weather and traffic).
3. **State ($S_t$):** User context (dietary preferences, order history, price sensitivity), spatial context (GPS distance, rain/traffic), and operational context (kitchen load, rider fleet availability).
4. **Actions ($A_t$):** Deciding the ranked sequence of top-N restaurant cards, carousel banners, and discount highlights displayed on the user's home screen.
5. **Rewards ($R_t$):**
   - $+5.0$ if the user completes an order.
   - $+2.0$ if food is delivered within the promised ETA.
   - $+3.0$ if user gives a 4/5-star rating or reorders within 7 days.
   - $-3.0$ if user bounces without scrolling.
   - $-5.0$ if cart is abandoned due to high fees or delivery latency.
   - $-10.0$ if the order is canceled due to kitchen/rider failure.

In [2]:
# Zomato Recommendation Environment simulation
class ZomatoRecommendationEnv:
    def __init__(self):
        self.restaurants = ["Biryani Central", "Pizza Hub", "Green Salad Bar", "Burger Junction"]
        
    def step(self, action_idx):
        selected = self.restaurants[action_idx]
        # Simulated operational outcome: 70% completed on time, 15% delayed, 15% bounce
        outcome = np.random.choice(["success", "delayed", "bounce"], p=[0.7, 0.15, 0.15])
        if outcome == "success":
            reward = 7.0  # +5 order, +2 on-time
        elif outcome == "delayed":
            reward = 0.0  # +5 order, -5 delay penalty
        else:
            reward = -3.0 # bounce penalty
        return selected, reward

import numpy as np
env = ZomatoRecommendationEnv()
restaurant, reward = env.step(action_idx=0)
print(f"Recommended: {restaurant} | Step Reward: {reward}")

Recommended: Biryani Central | Step Reward: 7.0


### Question 3
Imagine you are designing a simple RL agent for a cricket game app (like IPL fantasy league). Describe what could be the agent, environment, possible actions, and how rewards would be assigned for the agent's choices.

- **Agent:** An algorithmic fantasy squad manager optimizing player picks and multiplier designations across tournament fixtures.
- **Environment:** The fantasy league game system, including player pool statistics, match conditions (pitch report, ground dimensions, weather), confirmed playing XI (announced after the toss), and a 100-credit budget cap.
- **State ($S_t$):** Remaining budget credits out of 100, roster positional constraints filled (1-4 WKs, 3-6 Batsmen, 1-4 All-Rounders, 3-6 Bowlers; max 7 per IPL franchise), pitch characteristics, and confirmed Playing XI rosters.
- **Possible Actions ($A_t$):** Drafting an 11-player squad within credit/role limits, selecting the Captain ($2\times$ multiplier), selecting the Vice-Captain ($1.5\times$ multiplier), and making transfers between matches.
- **Reward Assignment ($R_t$):**
  - Positive rewards directly equal fantasy points scored by selected players (runs, boundaries, wickets, catches, maiden overs).
  - Captain and Vice-Captain bonus point multipliers ($2\times$ and $1.5\times$).
  - $-50$ penalty for selecting a benched player.
  - $-20$ penalty for violating budget or team composition limits.
  - Negative points for high bowling economy (>12.0) or ducks.

In [4]:
# IPL Fantasy Selection logic simulation
def evaluate_fantasy_pick(player_name, role, cost, points_earned, is_captain=False, is_playing=True):
    if not is_playing:
        return -50.0  # Penalty for drafting unannounced player
    if cost > 15.0:
        return -20.0  # Penalty for exceeding individual player budget rule
    
    multiplier = 2.0 if is_captain else 1.0
    return points_earned * multiplier

reward_sample = evaluate_fantasy_pick("Virat Kohli", "Batsman", cost=10.5, points_earned=78, is_captain=True, is_playing=True)
print(f"Fantasy Reward for Captain Pick: {reward_sample} points")

Fantasy Reward for Captain Pick: 156.0 points


### Question 4
Think of a feature in any app you use (like YouTube autoplay, Flipkart product suggestions, or Spotify playlist generation) that could be improved using Reinforcement Learning. Briefly describe how RL could make the feature smarter, and what the reward signal might be.

Traditional YouTube autoplay optimizes greedily for immediate click-through or short-term watch time on the next video, which frequently traps users in narrow, repetitive loops and induces session fatigue.

An RL formulation treats autoplay as a sequential Markov Decision Process (MDP) across an entire viewing session. By anticipating future states rather than greedily picking single videos, the agent balances viewer mood familiarity with serendipitous topic discovery, prolonging total session satisfaction.

The reward signal ($R_t$) evaluates user retention and engagement at the end of each autoplayed video:
$$R_t = w_1 \cdot \text{WatchPct} + w_2 \cdot \text{Engagement} - w_3 \cdot \text{SkipPenalty} - w_4 \cdot \text{ExitPenalty}$$
- $+1.0$ if the user watches $\ge 75\%$ of the video.
- $+2.0$ if the user likes, shares, or saves the video to a playlist.
- $-1.5$ if the user skips within the first 10 seconds.
- $-3.0$ if the user closes the app immediately after the video starts (session termination penalty).

In [5]:
# Simulation of an Epsilon-Greedy RL Agent for YouTube Autoplay Session Planning
class AutoplayAgent:
    def __init__(self, num_categories=4, epsilon=0.15):
        self.k = num_categories
        self.epsilon = epsilon
        self.q_values = np.zeros(num_categories)
        self.action_counts = np.zeros(num_categories)
        
    def select_action(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k)
        return int(np.argmax(self.q_values))
        
    def update(self, action, reward):
        self.action_counts[action] += 1
        alpha = 1.0 / self.action_counts[action]
        self.q_values[action] += alpha * (reward - self.q_values[action])

categories = ["Direct Follow-up", "Related Topic", "New Discovery", "Trending"]
agent = AutoplayAgent(num_categories=len(categories), epsilon=0.2)

print("Simulating 10 Autoplay Decisions:")
for step in range(1, 11):
    action = agent.select_action()
    simulated_reward = float(np.random.choice([2.0, 1.0, 0.2, -1.5], p=[0.3, 0.4, 0.1, 0.2]))
    agent.update(action, simulated_reward)
    print(f"Step {step:2d} | Suggested: {categories[action]:16s} | Received Reward: {simulated_reward:4.1f}")

print("\nFinal Estimated Action-Value (Q-values):")
for cat, q in zip(categories, agent.q_values):
    print(f"- {cat:16s}: {q:+.3f}")

Simulating 10 Autoplay Decisions:
Step  1 | Suggested: Direct Follow-up | Received Reward:  1.0
Step  2 | Suggested: Direct Follow-up | Received Reward: -1.5
Step  3 | Suggested: Related Topic    | Received Reward:  1.0
Step  4 | Suggested: Related Topic    | Received Reward:  1.0
Step  5 | Suggested: Related Topic    | Received Reward:  1.0
Step  6 | Suggested: Direct Follow-up | Received Reward:  1.0
Step  7 | Suggested: Direct Follow-up | Received Reward:  1.0
Step  8 | Suggested: New Discovery    | Received Reward:  1.0
Step  9 | Suggested: Related Topic    | Received Reward:  1.0
Step 10 | Suggested: Related Topic    | Received Reward: -1.5

Final Estimated Action-Value (Q-values):
- Direct Follow-up: +0.375
- Related Topic   : +0.500
- New Discovery   : +1.000
- Trending        : +0.000
